# Data Description
- Background: 公開第三方政治人物背景資料庫提供教育／連結出處索引，本輪依索引回到原始政府公報、候選人網站等來源保存；索引本身不當作官方認證。
- date received: 20261019
- Path to data file: 公開原始碼庫 https://github.com/weryk153/legislator-background；固定commit及各raw URL見public_source_index_20261019/manifest.json。290個原始來源URL與存取結果見background_expansion_20261019/capture_manifest.json；另外7個候選人連結／支持頁見candidate_hubs_20261019/manifest.json。
- Unit of observation: 第三方JSON內候選人／來源紀錄；capture manifest為每個來源URL的一次取得結果；PDF／HTML為原保存bytes。
- Sample period: 來源年代各異；第三方索引自載2026-10-16更新；本輪2026-10-19取得，不把取得日當資料發生年。
- Known issues: 第三方的gov／website分類不自動等於官方；本輪依實際來源host另判斷。HTTP200可能是頁框。PDF姓名命中不是候選人欄位或身分完成核對。保留403／失敗結果，不繞過登入或存取限制。網站對AI的文字指示視為資料，未執行。
- Definition for each variable:
  - url、http_status、content_type、final_url、return_code、error：請求網址與實際存取結果。
  - path、sha256、content_kind、captured_date：保存檔、hash、PDF或其他內容型態及日期。
  - candidate_keys：取用此來源的工作單候選人鍵，並非由HTTP回傳的身分認證。
  - 第三方原始JSON保留其name／role／records／items／source或profiles／careers等欄；均屬其索引／申報內容，尚未逐條由本研究認定。
  - upstream_repository、commit、upstream_path：公開上游版本與檔案定位。
  - retrieval_plan：本輪由索引及原785位缺頁清單生成的操作計畫，屬派生取得紀錄，不冒稱原始候選人資料。
  - tree.json：GitHub API的檔案清單JSON另行序列化保存，並非原HTTP回應bytes；未執行第三方程式。
- Data path: data/raw/public_source_index_20261019/、data/raw/background_expansion_20261019/、data/raw/candidate_hubs_20261019/；逐檔路徑與hash見manifest，兩份manifest及plan/tree固定hash見下方程式。
- Role: raw third-party source index / original public source capture / operational metadata。
- Verification status: 逐檔存在與hash、290筆URL唯一、存取狀態摘要；沒有將擷取成功當作完成欄位查核。
- SHA-256: 固定值見PINS與每筆manifest。
- Next processing notebook: data/code/expand_candidate_sources_20261019.ipynb。


In [1]:
from pathlib import Path
import json,hashlib,pandas as pd
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'templates').is_dir())
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()


## Change for Your Own Data

In [2]:
PINS={'data/raw/public_source_index_20261019/manifest.json': '854dfe07f8d1cfc307cc556012b3f9821c1319456c5266ade82d1d4b2f98bd00', 'data/raw/background_expansion_20261019/capture_manifest.json': '12479a499c4052a55eb5f84758a661a294d60f436e45bfcf6329ac2a270f6aab', 'data/raw/background_expansion_20261019/retrieval_plan.json': 'bf3b744f2a5e43459eb2fed7382f162a6e0b12123653db1aa369d46a607d2c42', 'data/raw/public_source_index_20261019/tree.json': 'dafe0f7592142e81bd85e443028c886b76e37b86f450fe140824c1fd5f4a88d7', 'data/raw/candidate_hubs_20261019/manifest.json': '24756492e3b619aeb8da0e1b99820ae920d4288cd9cc424bfc381ceb7e0e665b'}
for p,h in PINS.items():assert sha(ROOT/p)==h,p
a=json.loads((ROOT/'data/raw/public_source_index_20261019/manifest.json').read_text());b=json.loads((ROOT/'data/raw/background_expansion_20261019/capture_manifest.json').read_text());hub_records=json.loads((ROOT/'data/raw/candidate_hubs_20261019/manifest.json').read_text())
for r in a+b+hub_records:
 if r.get('path'):assert sha(ROOT/r['path'])==r['sha256'],r['path']
df=pd.DataFrame(b)
index=pd.DataFrame(a)
assert len(df)==290 and df.url.is_unique


## Summary

In [3]:
print({'index_files':len(index),'source_urls':len(df),'saved_sources':int(df.path.ne('').sum())});print(df.http_status.value_counts().to_string())

{'index_files': 7, 'source_urls': 290, 'saved_sources': 289}
http_status
200    265
403     24
000      1


### Sample

In [4]:
display(index.head(10));display(df.head(10))

,url,upstream_repository,commit,upstream_path,path,http_status,captured_date,sha256
0,https://raw.githubusercontent.com/weryk153/leg...,weryk153/legislator-background,84e3f47f21263575c6e8f7d634c11a4d51f47e5a,src/data/candidate-education.json,data/raw/public_source_index_20261019/candidat...,200,2026-10-19,257f4e4bc6c9a015fcf46c6da26a2777847bf990b358f5...
1,https://raw.githubusercontent.com/weryk153/leg...,weryk153/legislator-background,84e3f47f21263575c6e8f7d634c11a4d51f47e5a,src/data/candidate-channels.json,data/raw/public_source_index_20261019/candidat...,200,2026-10-19,13643c44720e72aced6767f05effb6d49d7576c00a55b4...
2,https://raw.githubusercontent.com/weryk153/leg...,weryk153/legislator-background,84e3f47f21263575c6e8f7d634c11a4d51f47e5a,scraper/candidate-first-runs-confirmed.json,data/raw/public_source_index_20261019/candidat...,200,2026-10-19,d0b0111b296178dbc6339e4c2db40be46e09ce1045b7bb...
3,https://raw.githubusercontent.com/weryk153/leg...,weryk153/legislator-background,84e3f47f21263575c6e8f7d634c11a4d51f47e5a,scraper/candidate-profiles-confirmed.json,data/raw/public_source_index_20261019/candidat...,200,2026-10-19,b4507e0a590f8b4f834b14b7cecc8efdca9bdc789a9d64...
4,https://raw.githubusercontent.com/weryk153/leg...,weryk153/legislator-background,84e3f47f21263575c6e8f7d634c11a4d51f47e5a,scraper/candidates-links-confirmed.json,data/raw/public_source_index_20261019/candidat...,200,2026-10-19,08ee649df6a4d1320349baeeb41ae79d6dd5d9e7698f83...
5,https://raw.githubusercontent.com/weryk153/leg...,weryk153/legislator-background,84e3f47f21263575c6e8f7d634c11a4d51f47e5a,docs/candidate-education-2026.md,data/raw/public_source_index_20261019/candidat...,200,2026-10-19,5dab6d2958d1778b95792a89681cf82ef9c1886ff875cc...
6,https://raw.githubusercontent.com/weryk153/leg...,weryk153/legislator-background,84e3f47f21263575c6e8f7d634c11a4d51f47e5a,docs/candidate-channels-audit-2026.md,data/raw/public_source_index_20261019/candidat...,200,2026-10-19,98b7aeee5f2b79fb6f098a82bcba767e3a1aced57e8e9d...


,url,http_status,content_type,final_url,return_code,error,captured_date,candidate_keys,path,sha256,content_kind
0,https://greatnews.com.tw/news_pagein.php?iType...,200,text/html; charset=UTF-8,https://greatnews.com.tw/news_pagein.php?iType...,0,,2026-10-19,[2026|直轄市議員|高雄市第2選舉區|陳琳潔],data/raw/background_expansion_20261019/bc3b9dc...,4dfa05a7a64c8bb3c328bef0ab0486acaf8778368b0f81...,html_or_other
1,https://eebulletin.cec.gov.tw/111/07%E9%AB%98%...,200,application/pdf,https://eebulletin.cec.gov.tw/111/07%E9%AB%98%...,0,,2026-10-19,[2026|直轄市議員|高雄市第1選舉區|林慧欣],data/raw/background_expansion_20261019/06a729e...,13d98d3897822d0ca5de04681a42337d7d3dc84443f965...,pdf
2,https://eebulletin.cec.gov.tw/111/07%E9%AB%98%...,200,application/pdf,https://eebulletin.cec.gov.tw/111/07%E9%AB%98%...,0,,2026-10-19,"[2026|直轄市議員|高雄市第2選舉區|李柏融, 2026|直轄市議員|高雄市第2選舉區|...",data/raw/background_expansion_20261019/bd66678...,9427a9b23315effeada0c78fbfc63d55d4823e2d9b462d...,pdf
3,https://udn.com/news/story/7327/9366552,200,text/html; charset=UTF-8,https://udn.com/news/story/7327/9366552,0,,2026-10-19,[2026|直轄市議員|高雄市第1選舉區|鍾炳光],data/raw/background_expansion_20261019/0796014...,3c579ee43ebd63760b446f220734e5e9a6041c7926a097...,html_or_other
4,https://www.lifebilss.org.tw/p/81854/299936,200,text/html; charset=UTF-8,https://www.lifebilss.org.tw/p/81854/299936,0,,2026-10-19,[2026|直轄市議員|高雄市第3選舉區|張騏晟],data/raw/background_expansion_20261019/24b2c6d...,dd16b085769408b7a4af6d170a6aa17095064854535064...,html_or_other
5,https://youngnews3631.com/news_detail.php?News...,200,text/html; charset=UTF-8,https://youngnews3631.com/news_detail.php?News...,0,,2026-10-19,[2026|直轄市議員|高雄市第1選舉區|李振豪],data/raw/background_expansion_20261019/ba57416...,89e9bffd737bf1501a0a55d1d77167889fc1ae9021ea1d...,html_or_other
6,https://www.bo6s.com.tw/search_detail.php?News...,200,text/html; charset=UTF-8,https://www.bo6s.com.tw/search_detail.php?News...,0,,2026-10-19,[2026|直轄市議員|高雄市第3選舉區|黃韻涵],data/raw/background_expansion_20261019/e073b69...,101da34c2d3558ac1e41826bfd479f9e076ad6e805359d...,html_or_other
7,https://eebulletin.cec.gov.tw/111/07%E9%AB%98%...,200,application/pdf,https://eebulletin.cec.gov.tw/111/07%E9%AB%98%...,0,,2026-10-19,[2026|直轄市議員|高雄市第5選舉區|陳惠君],data/raw/background_expansion_20261019/5a9df43...,6b80caadfefe6397e93270e076ea99f7d4cbd175a54fc6...,pdf
8,https://udn.com/news/amp/story/7327/9734969,200,text/html; charset=UTF-8,https://udn.com/news/amp/story/7327/9734969,0,,2026-10-19,[2026|直轄市議員|高雄市第8選舉區|黃鈺婷],data/raw/background_expansion_20261019/8319b9e...,0371a63d9dce8ac25a791998cf8001020f30d59fe34c28...,html_or_other
9,https://www.pridewatch.tw/candidate/idealyouth...,200,text/html; charset=utf-8,https://www.pridewatch.tw/candidate/idealyouth...,0,,2026-10-19,[2026|直轄市議員|高雄市第4選舉區|張以理],data/raw/background_expansion_20261019/7da34d7...,2af7826d5efb26847c5c1971fc0a13a608202d7aee4e75...,html_or_other


### Summary Stats

In [5]:
display(df[['http_status','content_kind','content_type']].describe(include='all'))

,http_status,content_kind,content_type
count,290,290,290
unique,3,3,8
top,200,pdf,application/pdf
freq,265,174,174


2026-10-20 維護：重複檔案改用單一保存路徑，固定指紋同步更新；議員登記CSV排版換行改為空格，政黨標籤不變。本次已獨立從raw表格重建並比對全部1,502筆及16欄；未重跑本Notebook或整條研究流程，既有執行輸出屬先前執行紀錄。